In [0]:
from pyspark.sql.functions import col,initcap,concat_ws,lit

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run /Workspace/Users/chrknov6@hotmail.com/formula1/Incremental/00.Configurations

In [0]:
%run "/Workspace/Users/chrknov6@hotmail.com/formula1/Incremental/002.silver helper functions"

In [0]:
bronze_table = f'{catalog}.{bronze_schema}.drivers'
silver_table = f'{catalog}.{silver_schema}.drivers'

In [0]:
drivers_df = (spark.table(bronze_table)
                   .filter(col("batch_id") == lit(v_batch_id))
                   .drop(col("url"))
                   .withColumnsRenamed(
                       {
                           "driverId":"driver_id",
                           "dateOfBirth":"date_of_birth"
                       })
                   .withColumn("driver_name",initcap(concat_ws(" ",col("givenName"),col("familyName"))))
                   .drop(col("givenName"),col("familyName"))
                   .dropDuplicates(["driver_id"])
                   .withColumn("nationality",initcap(col("nationality")))
              )

In [0]:
write_to_silver(
    input_df= drivers_df,
    table_name= silver_table,
    merge_condition=col("t.driver_id") == col("s.driver_id"),
    columns_to_update= ["date_of_birth","driver_name","nationality","ingestion_time","filename"]
)